In [1]:
# import required libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)
print("Pandas version:", pd.__version__)

TensorFlow version: 2.19.0
Pandas version: 2.2.2


In [3]:
# Step 1: Create a small synthetic "customer churn" dataset
# Each row ~ one customer
# Columns:
#   - age            : integer (years)
#   - monthly_spend  : float (how much they pay per month)
#   - tenure_months  : integer (how long they've been a customer)
#   - churn          : target (0 = stayed, 1 = left)

np.random.seed(42)  # for reproducibility

n_samples = 1000  # number of customers

# Generate random feature values with reasonable ranges
age = np.random.randint(18, 70, size=n_samples)
monthly_spend = np.random.uniform(10, 200, size=n_samples)
tenure_months = np.random.randint(1, 60, size=n_samples)

# Define a simple "churn probability" rule:
# - customers who spend a lot AND have small tenure are more likely to churn
# - customers with long tenure are less likely to churn
#   (this is just to create a pattern for the model to learn)
churn_prob = (
    0.3 * (monthly_spend / monthly_spend.max()) +          # higher spend => somewhat more churn
    0.7 * (1 - tenure_months / tenure_months.max())        # lower tenure => more churn
)
# Ensure probabilities are in [0, 1]
churn_prob = np.clip(churn_prob, 0, 1)

# Sample churn as 0/1 according to churn_prob (Bernoulli trials)
churn = np.random.binomial(1, churn_prob)

# Build a pandas DataFrame (like reading a real CSV)
df = pd.DataFrame({
    "age": age,
    "monthly_spend": monthly_spend,
    "tenure_months": tenure_months,
    "churn": churn
})

In [4]:
# checking the dataset (Data Analysis)
df.head() # shows first 5 rows

,age,monthly_spend,tenure_months,churn
0,56,111.197281,28,1
1,69,56.069396,51,0
2,46,61.156214,40,0
3,32,81.683991,54,1
4,60,13.813528,5,1


In [5]:
df.shape # gives (num of rows, num columns)

(1000, 4)

In [6]:
df.info()  # shows data types and missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   age            1000 non-null   int64  
 1   monthly_spend  1000 non-null   float64
 2   tenure_months  1000 non-null   int64  
 3   churn          1000 non-null   int64  
dtypes: float64(1), int64(3)
memory usage: 31.4 KB


In [7]:
df.describe() # gives min, max, mean, standard deviation(std), percentile for numeric columns

,age,monthly_spend,tenure_months,churn
count,1000.00000,1000.000000,1000.000000,1000.000000
mean,43.81900,104.657160,29.586000,0.512000
std,14.99103,55.047937,16.826043,0.500106
min,18.00000,10.880084,1.000000,0.000000
25%,31.00000,55.056169,16.000000,0.000000
50%,44.00000,105.817862,29.000000,1.000000
75%,56.00000,150.246128,44.000000,1.000000
max,69.00000,199.888608,59.000000,1.000000


In [8]:
# Step 3: Split into features (X) and target (y)

feature_cols = ["age", "monthly_spend", "tenure_months"]  # input features
target_col = "churn"                                     # target label

X = df[feature_cols].values.astype("float32")  # shape (N, 3)
y = df[target_col].values.astype("int32")      # shape (N,)

print("X shape:", X.shape)  # (num_samples, num_features)
print("y shape:", y.shape)  # (num_samples,)
print("First 5 feature rows:\n", X[:5])
print("First 5 labels:", y[:5])

X shape: (1000, 3)
y shape: (1000,)
First 5 feature rows:
 [[ 56.       111.19728   28.      ]
 [ 69.        56.069397  51.      ]
 [ 46.        61.156216  40.      ]
 [ 32.        81.68399   54.      ]
 [ 60.        13.813527   5.      ]]
First 5 labels: [1 0 0 1 1]


In [9]:
# Step 4: Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% test, 80% train (common choice)
    random_state=42,    # for reproducibility
    stratify=y          # keep class balance similar in train and test
)

print("Train X shape:", X_train.shape)
print("Train y shape:", y_train.shape)
print("Test X shape:", X_test.shape)
print("Test y shape:", y_test.shape)

Train X shape: (800, 3)
Train y shape: (800,)
Test X shape: (200, 3)
Test y shape: (200,)


In [10]:
# Step 5: Normalize features using StandardScaler (mean 0, std 1 per feature)
# Neural nets train better when features are on a similar scale.

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit scaler on training data only, then transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled X_train mean (approx):", X_train_scaled.mean(axis=0))
print("Scaled X_train std (approx):", X_train_scaled.std(axis=0))

Scaled X_train mean (approx): [1.77323827e-08 1.69873235e-08 1.25076625e-08]
Scaled X_train std (approx): [0.9999999 1.0000001 0.9999999]


In [11]:
# Step 6: Build the Keras model (Functional API, but still simple)
# small neural network: input → 2 hidden layers → sigmoid output.

# Input shape is (num_features,) = (3,)
inputs = tf.keras.Input(shape=(X_train_scaled.shape[1],))  # (3,)

# Hidden layer 1: 16 neurons with ReLU activation
x = layers.Dense(16, activation="relu")(inputs)

# Hidden layer 2: 8 neurons with ReLU activation
x = layers.Dense(8, activation="relu")(x)

# Output layer: 1 neuron with sigmoid -> probability of churn (class 1)
outputs = layers.Dense(1, activation="sigmoid")(x)

tabular_model = tf.keras.Model(inputs=inputs, outputs=outputs)

tabular_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 3)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209 (836.00 B)

 Trainable params: 209 (836.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Step 7: Compile the model
# For binary classification:
#   - loss = BinaryCrossentropy
#   - metrics = ["accuracy"] is a common start

tabular_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"]
)

# Train the model with a validation split from the training data
history_tabular = tabular_model.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    batch_size=32,         # # model will look at 32 training examples at a time before doing one weight update.
    validation_split=0.2,  # 20% of training data used for validation
    verbose=2
)

Epoch 1/20
20/20 - 6s - 322ms/step - accuracy: 0.6078 - loss: 0.6764 - val_accuracy: 0.6313 - val_loss: 0.6693
Epoch 2/20
20/20 - 1s - 28ms/step - accuracy: 0.6406 - loss: 0.6666 - val_accuracy: 0.6313 - val_loss: 0.6610
Epoch 3/20
20/20 - 1s - 31ms/step - accuracy: 0.6656 - loss: 0.6581 - val_accuracy: 0.6500 - val_loss: 0.6536
Epoch 4/20
20/20 - 0s - 24ms/step - accuracy: 0.6687 - loss: 0.6495 - val_accuracy: 0.6625 - val_loss: 0.6458
Epoch 5/20
20/20 - 1s - 30ms/step - accuracy: 0.6734 - loss: 0.6411 - val_accuracy: 0.6562 - val_loss: 0.6380
Epoch 6/20
20/20 - 1s - 28ms/step - accuracy: 0.6781 - loss: 0.6325 - val_accuracy: 0.6562 - val_loss: 0.6309
Epoch 7/20
20/20 - 1s - 25ms/step - accuracy: 0.6828 - loss: 0.6248 - val_accuracy: 0.6250 - val_loss: 0.6251
Epoch 8/20
20/20 - 1s - 25ms/step - accuracy: 0.6875 - loss: 0.6189 - val_accuracy: 0.6375 - val_loss: 0.6195
Epoch 9/20
20/20 - 0s - 21ms/step - accuracy: 0.6875 - loss: 0.6138 - val_accuracy: 0.6562 - val_loss: 0.6154
Epoch 10/

In [13]:
# Step 8: Evaluate on the held-out test set

test_loss, test_acc = tabular_model.evaluate(X_test_scaled, y_test, verbose=2)
print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

7/7 - 0s - 38ms/step - accuracy: 0.5950 - loss: 0.6492
Test loss: 0.6492255330085754
Test accuracy: 0.5950000286102295
